# 🌶️ Prueba del detector de locotes en Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Legoshi7755/deteccion-locotes/blob/main/colab_prueba_locotes.ipynb)

Este notebook usa la GPU gratuita de Colab para probar el modelo `best_frutos_video_ampliado.pt` (YOLO de segmentación, clase `locote_rojo`) con los mismos umbrales y filtros que la aplicación WATSOCOTE.

| Parte | Qué hace | Necesita |
|---|---|---|
| **1. Fotos** | Detecta y segmenta locotes en fotos | Fotos tuyas |
| **2. Video** | Anota un video completo y lo descarga | Un video grabado |
| **3. En vivo** *(experimental)* | Corre la aplicación estéreo con los dos celulares | Los dos celulares transmitiendo |

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. Después ejecuta las celdas en orden con ▶ (o `Shift+Enter`).

## 0. Preparación

In [ ]:
#@title Verificar la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Sin GPU: cambia el tipo de entorno a 'T4 GPU' (funciona igual en CPU, pero más lento)."

In [ ]:
#@title Descargar el repositorio e instalar dependencias
# Colab ya trae PyTorch con CUDA y OpenCV; solo falta Ultralytics.
%cd /content
!test -d deteccion-locotes || git clone --depth 1 https://github.com/Legoshi7755/deteccion-locotes
%cd /content/deteccion-locotes
!git pull -q
!pip install -q ultralytics==8.4.191

In [ ]:
#@title Verificar la integridad de los archivos (SHA256)
import hashlib, json
from pathlib import Path
hashes = json.loads(Path("SHA256.json").read_text())
malos = [f for f, h in hashes.items() if not Path(f).is_file() or hashlib.sha256(Path(f).read_bytes()).hexdigest() != h]
print("✅ Los", len(hashes), "archivos coinciden con SHA256.json" if not malos else f"⚠️ No coinciden: {malos}")

In [ ]:
#@title Cargar el modelo
CONFIANZA = 0.40  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
import sys
from pathlib import Path
sys.path.insert(0, "/content/deteccion-locotes/fotos")
import prueba_imagenes as pi
pi.cargar_modelo(conf=CONFIANZA)

def ruta_drive(texto):
    # Acepta '/content/drive/MyDrive/Claude/Fotos', 'MyDrive/Claude/Fotos' o 'Claude/Fotos'.
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    texto = texto.strip().strip("/")
    for prefijo in ("content/drive/MyDrive/", "content/drive/", "MyDrive/", "Mi unidad/"):
        if texto.startswith(prefijo):
            texto = texto[len(prefijo):]
    ruta = Path("/content/drive/MyDrive") / texto
    if not ruta.exists():
        raise FileNotFoundError(f"No encontré '{texto}' en tu Drive. Escribe la ruta desde 'Mi unidad', "
                                f"por ejemplo: Claude/Fotos_locote")
    print("Usando:", ruta)
    return ruta

## 1. Probar con fotos

Sube una o varias fotos desde tu computadora o celular. Si prefieres usar fotos de tu Google Drive, escribe en `CARPETA_DRIVE` la ruta de la carpeta **desde "Mi unidad"**, por ejemplo `Claude/Fotos_locote` (también sirve la ruta completa `/content/drive/MyDrive/Claude/Fotos_locote`). Déjalo vacío para subir fotos desde tu equipo.

`USAR_SAM` refina el contorno con MobileSAM, como en la captura puntual de la aplicación (más preciso, más lento).

In [ ]:
#@title Subir y analizar fotos
CARPETA_DRIVE = "Claude/Fotos_locote"  #@param {type:"string"}
USAR_SAM = False  #@param {type:"boolean"}
import cv2, matplotlib.pyplot as plt
from pathlib import Path

entrada = Path("/content/entradas"); entrada.mkdir(exist_ok=True)
salida = Path("/content/resultados"); salida.mkdir(exist_ok=True)
if CARPETA_DRIVE:
    fotos = [p for p in sorted(ruta_drive(CARPETA_DRIVE).rglob("*")) if p.suffix.lower() in pi.EXT_IMAGEN]
else:
    from google.colab import files
    fotos = []
    for nombre, datos in files.upload().items():
        (entrada / nombre).write_bytes(datos)
        if Path(nombre).suffix.lower() in pi.EXT_IMAGEN:
            fotos.append(entrada / nombre)
print(f"{len(fotos)} foto(s)")
if not fotos:
    print("⚠️ No hay fotos .jpg/.png en esa carpeta.")

for ruta in fotos:
    imagen = cv2.imread(str(ruta))
    if imagen is None:
        print(ruta.name, "no se pudo leer"); continue
    vista, dets = pi.detectar(imagen, sam=USAR_SAM)
    cv2.imwrite(str(salida / f"{ruta.stem}_anotado.jpg"), vista)
    plt.figure(figsize=(9, 9)); plt.imshow(cv2.cvtColor(vista, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{ruta.name}: {len(dets)} locote(s) " + " ".join(f"{d['confianza']:.2f}" for d in dets)); plt.show()

In [ ]:
#@title Descargar las fotos anotadas (zip)
!cd /content && zip -qr resultados_fotos.zip resultados -i '*.jpg'
from google.colab import files
files.download("/content/resultados_fotos.zip")

## 2. Probar con un video

Graba un video del cultivo con el celular y súbelo, o escribe su ruta en Drive desde "Mi unidad" (por ejemplo `Claude/videos/cultivo.mp4`). Con `CONFIRMAR_DOS_CUADROS` un locote solo se marca si aparece en dos cuadros seguidos, igual que la detección continua de la aplicación; desactívalo para ver todas las detecciones crudas.

`ANALIZAR_CADA` = 1 analiza todos los cuadros; 2 o 3 lo hace más rápido en videos largos.

In [ ]:
#@title Subir y anotar un video
RUTA_VIDEO_DRIVE = ""  #@param {type:"string"}
ANALIZAR_CADA = 1  #@param {type:"integer"}
CONFIRMAR_DOS_CUADROS = True  #@param {type:"boolean"}
MAX_SEGUNDOS = 60  #@param {type:"integer"}
from pathlib import Path
if RUTA_VIDEO_DRIVE:
    video = ruta_drive(RUTA_VIDEO_DRIVE)
else:
    from google.colab import files
    subido = files.upload()
    nombre = next(iter(subido))
    video = Path("/content/entradas") / nombre
    video.parent.mkdir(exist_ok=True); video.write_bytes(subido[nombre])
Path("/content/resultados").mkdir(exist_ok=True)
crudo = Path("/content/resultados") / f"{video.stem}_crudo.mp4"
anotado = Path("/content/resultados") / f"{video.stem}_anotado.mp4"
pi.procesar_video(video, crudo, cada=max(1, ANALIZAR_CADA), confirmar=CONFIRMAR_DOS_CUADROS,
                  max_segundos=MAX_SEGUNDOS or None)
# Convertir a H.264 para que se vea en el navegador
!ffmpeg -y -loglevel error -i "{crudo}" -c:v libx264 -pix_fmt yuv420p -crf 23 "{anotado}" && rm "{crudo}"
print("Listo:", anotado)

In [ ]:
#@title Ver el video anotado
from base64 import b64encode
from IPython.display import HTML, display
tam = anotado.stat().st_size / 1e6
if tam > 60:
    print(f"El video pesa {tam:.0f} MB; descárgalo con la celda siguiente.")
else:
    display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{b64encode(anotado.read_bytes()).decode()}" type="video/mp4"></video>'))

In [ ]:
#@title Descargar el video anotado
from google.colab import files
files.download(str(anotado))

## 3. Aplicación en vivo con los dos celulares *(experimental)*

Corre la aplicación completa (`fotos/yolo_segmentacion_video_ampliado.py`) en Colab con GPU. El flujo:

```
Celulares ──VDO.Ninja──▶ tu navegador (página de la app) ──cuadros──▶ servidor en Colab (YOLO en GPU)
```

1. Ejecuta la celda **Iniciar servidor** y espera a que diga que está listo.
2. En **cada celular**, abre su enlace de emisión (se imprimen abajo), permite la cámara trasera e inicia la transmisión. Mantén la pantalla encendida.
3. Ejecuta **Abrir la aplicación** y entra al enlace que aparece. Ábrelo **en Chrome con la misma cuenta de Google** con la que usas Colab.

⚠️ **Ten en cuenta:**
- Cada par de imágenes viaja por internet hasta Colab: la detección continua será más lenta que en una PC local, según tu conexión.
- La calibración incluida solo sirve para el montaje con el que se obtuvo (posición, lente y zoom de los celulares).
- Colab apaga la máquina tras un rato de inactividad. Si activas "guardar datos de entrenamiento" en la app, las fotos quedan en `/content/deteccion-locotes/fotos/dataset_carrito_pendiente`, y **se pierden al cerrar Colab** salvo que las copies a Drive (celda al final).

In [ ]:
#@title Iniciar servidor
import subprocess, sys, time
from pathlib import Path
%cd /content/deteccion-locotes
log = Path("/content/servidor.log")
# Cerrar cualquier servidor anterior que siga ocupando el puerto 8766
subprocess.run(["pkill", "-f", "yolo_segmentacion_video_ampliado"])
time.sleep(1)
servidor = subprocess.Popen([sys.executable, "-u", "fotos/yolo_segmentacion_video_ampliado.py"],
                            stdout=log.open("w"), stderr=subprocess.STDOUT)
for _ in range(120):
    time.sleep(1)
    if "8766" in log.read_text() or servidor.poll() is not None:
        break
print(log.read_text())
print("✅ Servidor listo" if servidor.poll() is None else "❌ El servidor se detuvo: revisa el mensaje de arriba")

sys.path.insert(0, "/content/deteccion-locotes/fotos")
import configurar_video
print("\nEnlaces de emisión — abre cada uno EN SU CELULAR:")
for lado, url in configurar_video.enlaces().items():
    print(f"  {'Izquierda' if lado == 'izq' else 'Derecha'}: {url}")

In [ ]:
#@title Abrir la aplicación
from google.colab.output import eval_js, serve_kernel_port_as_window
url = eval_js("google.colab.kernel.proxyPort(8766)")
print("Aplicación:", url)
serve_kernel_port_as_window(8766, anchor_text="👉 Abrir WATSOCOTE en una pestaña nueva")

In [ ]:
#@title Ver el registro del servidor
print(Path("/content/servidor.log").read_text()[-4000:])

In [ ]:
#@title Copiar los datos de entrenamiento guardados a Google Drive
from google.colab import drive
import shutil
drive.mount("/content/drive")
origen = Path("/content/deteccion-locotes/fotos/dataset_carrito_pendiente")
if origen.exists():
    destino = Path("/content/drive/MyDrive/watsocote_dataset_pendiente")
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    print("Copiado a", destino)
else:
    print("No hay datos guardados (activa la opción en la aplicación).")

In [ ]:
#@title Detener el servidor
servidor.terminate(); servidor.wait(10)
print("Servidor detenido.")